In [1]:
import sys, time, gc, importlib
from pathlib import Path
import duckdb, numpy as np, pandas as pd, xgboost as xgb
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.blocking as B, src.features as F, src.metrics as M

ARTIFACTS = ROOT / "artifacts"
PARQ, KEYS2 = ARTIFACTS / "parquet", ARTIFACTS / "keys_v2"
def pq(name): return f"read_parquet('{(PARQ / f'{name}.parquet').as_posix()}')"
def k2(name): return f"read_parquet('{(KEYS2 / f'{name}.parquet').as_posix()}')"

con = duckdb.connect((ARTIFACTS / "blocking.duckdb").as_posix())
B.setup(con, threads=8, memory_limit="5GB", temp_dir=(ARTIFACTS / "duckdb_tmp").as_posix())
F.setup_macros(con)

S1_ALL = k2("train_source1")
TGT = f"(SELECT * FROM {k2('train_source2')} UNION ALL SELECT * FROM {k2('train_source3')})"

# rebuild eval features + v3 predictions
mx3 = xgb.XGBClassifier()
mx3.load_model(str(ARTIFACTS / "matcher_xgb_v3.json"))

t0 = time.time()
ev3 = F.build_features_chunked(con, "m_ev_rows", S1_ALL, TGT, n_chunks=2, feats=F.FEATS_V3)
ev3 = ev3.merge(con.execute("SELECT s1_id, tgt_id, y FROM m_ev_rows").fetchdf(), on=["s1_id", "tgt_id"])
ev3["q_v3"] = mx3.predict_proba(ev3[F.FEATS_V3])[:, 1]
print(f"ev3 {ev3.shape}  ({time.time() - t0:.0f}s) | AUC {roc_auc_score(ev3.y, ev3.q_v3):.4f}")

con.register("ev3q", ev3[["s1_id", "tgt_id", "q_v3", "y"]])
display(M.eval_f05(con, "(SELECT s1_id, tgt_id FROM ev3q WHERE q_v3 >= 0.7)", "v3 q>=0.7 (sanity)"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ev3 (132086, 62)  (14s) | AUC 0.9992


,method,macro_f05,f05_singletons,f05_matched,micro_precision,micro_recall,avg_pred
0,v3 q>=0.7 (sanity),0.980848,0.975678,0.98114,0.995115,0.956088,3.3268


In [2]:
con.execute("CREATE OR REPLACE TABLE miss_c AS SELECT e.* FROM eval_pairs e ANTI JOIN m_ev_rows USING (s1_id, tgt_id)")

display(con.execute(f"""
WITH m AS (SELECT m.s1_id, m.tgt_id, (c2.tgt_id IS NOT NULL) AS in_stage1
           FROM miss_c m LEFT JOIN cand2 c2 USING (s1_id, tgt_id)),
k1 AS (SELECT eid(entity_id) AS id, name_core, addr_v2 FROM {S1_ALL}),
kt AS (SELECT eid(entity_id) AS id, name_core, name_v2, addr_v2 FROM {TGT}),
found AS (SELECT s1_id, tgt_id FROM ev3q WHERE y = 1 AND q_v3 >= 0.7),
sib AS (SELECT m.s1_id, m.tgt_id,
               max(jaro_winkler_similarity(coalesce(a.name_core, ''), coalesce(b.name_core, ''))) AS sib_name_jw,
               max(jac(a.addr_v2, b.addr_v2)) AS sib_addr_jacc
        FROM m JOIN found f USING (s1_id)
        JOIN kt a ON a.id = m.tgt_id JOIN kt b ON b.id = f.tgt_id
        GROUP BY 1, 2)
SELECT CASE WHEN in_stage1 THEN 'dropped by reranker cut' ELSE 'never retrieved (stage 1)' END AS kind,
       count(*)                                                                        AS n,
       avg((t.addr_v2 IS NULL)::INT)                                                   AS tgt_addr_missing,
       avg((s.name_core = t.name_core)::INT)                                           AS exact_core_name,
       avg(jaro_winkler_similarity(coalesce(s.name_core, ''), coalesce(t.name_core, ''))) AS avg_name_jw_to_s1,
       avg(coalesce(jac(s.addr_v2, t.addr_v2), 0))                                     AS avg_addr_jacc_to_s1,
       avg((coalesce(sib.sib_name_jw, 0) >= 0.92 OR coalesce(sib.sib_addr_jacc, 0) >= 0.5)::INT) AS close_to_found_sibling
FROM m JOIN k1 s ON s.id = m.s1_id JOIN kt t ON t.id = m.tgt_id
LEFT JOIN sib USING (s1_id, tgt_id)
GROUP BY 1
""").fetchdf())

pd.set_option("display.max_colwidth", 45)
display(con.execute(f"""
SELECT s.name_v2 AS s1_name, t.name_v2 AS tgt_name, s.addr_v2 AS s1_addr, t.addr_v2 AS tgt_addr,
       s.country_key AS cty
FROM miss_c m
JOIN (SELECT eid(entity_id) AS id, * FROM {S1_ALL}) s ON s.id = m.s1_id
JOIN (SELECT eid(entity_id) AS id, * FROM {TGT})    t ON t.id = m.tgt_id
ORDER BY hash(m.s1_id + m.tgt_id) LIMIT 30
""").fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,kind,n,tgt_addr_missing,exact_core_name,avg_name_jw_to_s1,avg_addr_jacc_to_s1,close_to_found_sibling
0,dropped by reranker cut,189,0.068783,0.164021,0.686108,0.372197,0.788360
1,never retrieved (stage 1),1433,0.571528,0.187020,0.856019,0.158744,0.639218


,s1_name,tgt_name,s1_addr,tgt_addr,cty
0,tirupati ventures private limited,tirupativentures,2nd floor e 218b e block sector 63 noida ...,uttar pradesh 2nd floor e 218b e block se...,india
1,shree enterprises private limited,sri enterprises private limited,201 2nd floor nanik niwas plot no 13 oppo...,floor mumbai maharashtra,india
2,physical therapy partners,physical partners therapy,20600 55th street unit 574 phoenix az,NaN,us
3,paras traders limited,paras tdrnerts limited,rue no 3 laxmi niwas chawl janu pada vaib...,NaN,india
4,zephon hotels llc,zephon hotel llc,910 foster drive baton rouge city la,NaN,us
5,sree infotech private limited,umbrabelocalo,15 jatin das road kolkata kolkata howrah ...,15 howrah kolkata bengal,india
6,blair godinez mckinley lp,blairmckinleygodinez,1200 n capitol street unit a612 wa dc,n capitol street wa dc,us
7,t and k beyond,t and,18351 atlas street omaha ne,NaN,us
8,arts league,arts laneceu,12313 edenvale path tx austin,NaN,us
9,lotus care private limited,lotus care care,p 29 mission row extention east court kol...,NaN,india


In [4]:
fn3 = ev3[(ev3.y == 1) & (ev3.q_v3 < 0.7)][["s1_id", "tgt_id"]]
con.register("fn3", fn3)

In [5]:
display(con.execute(f"""
WITH kt AS (SELECT eid(entity_id) AS id, name_v2, addr_v2 FROM {TGT}),
comp AS (SELECT r.tgt_id, r.s1_id AS comp_s1
         FROM cut_rev r JOIN fn3 f ON f.tgt_id = r.tgt_id AND r.s1_id <> f.s1_id
         QUALIFY row_number() OVER (PARTITION BY r.tgt_id ORDER BY r.p DESC) = 1),
sibs AS (SELECT s1_id, tgt_id FROM cut_rev WHERE p >= 0.9),
true_sim AS (SELECT f.s1_id, f.tgt_id, max(jaro_winkler_similarity(a.name_v2, b.name_v2)) AS ts
             FROM fn3 f JOIN sibs s ON s.s1_id = f.s1_id AND s.tgt_id <> f.tgt_id
             JOIN kt a ON a.id = f.tgt_id JOIN kt b ON b.id = s.tgt_id GROUP BY 1, 2),
comp_sim AS (SELECT f.s1_id, f.tgt_id, max(jaro_winkler_similarity(a.name_v2, b.name_v2)) AS cs
             FROM fn3 f JOIN comp c ON c.tgt_id = f.tgt_id
             JOIN sibs s ON s.s1_id = c.comp_s1 AND s.tgt_id <> f.tgt_id
             JOIN kt a ON a.id = f.tgt_id JOIN kt b ON b.id = s.tgt_id GROUP BY 1, 2)
SELECT count(*) AS fn_with_competitor,
       avg((coalesce(ts, 0) > coalesce(cs, 0) + 0.02)::INT) AS true_closer,
       avg((coalesce(cs, 0) > coalesce(ts, 0) + 0.02)::INT) AS competitor_closer,
       avg((abs(coalesce(ts, 0) - coalesce(cs, 0)) <= 0.02)::INT) AS tie
FROM fn3 f JOIN comp c USING (tgt_id)
LEFT JOIN true_sim USING (s1_id, tgt_id) LEFT JOIN comp_sim USING (s1_id, tgt_id)
""").fetchdf())

,fn_with_competitor,true_closer,competitor_closer,tie
0,1004,0.489044,0.208167,0.300797
